# Icy Tower — trening PPO na Google Colab

1. **Runtime → Change runtime type → T4 GPU**
2. Uruchom komórki po kolei
3. Zapis na **Google Drive**: `MyDrive/icy_tower/`

Sklonuj repo z GitHuba **albo** ustaw `USE_UPLOAD = True` i wgraj zip w następnej komórce.

In [ ]:
# --- Konfiguracja ---
REPO_URL = "https://github.com/tomrat04/icy_tower.git"  # zmień na swoje repo
USE_UPLOAD = False  # True = wgraj icy_tower.zip ręcznie w komórce poniżej

DRIVE_DIR = "/content/drive/MyDrive/icy_tower"
N_ENVS = 24
TIMESTEPS = 10_000_000
RUN_NAME = "colab_ppo"

In [ ]:
!pip install -q pygame gymnasium "stable-baselines3[extra]" tensorboard tqdm rich

In [ ]:
import os
import shutil
from pathlib import Path

if USE_UPLOAD:
    from google.colab import files
    uploaded = files.upload()  # wybierz icy_tower.zip
    zip_name = next(iter(uploaded))
    !unzip -q -o "{zip_name}" -d /content
    if not Path("/content/icy_tower/train.py").exists():
        raise FileNotFoundError("W zipie brak folderu icy_tower/ z train.py")
else:
    if Path("/content/icy_tower").exists():
        shutil.rmtree("/content/icy_tower")
    !git clone {REPO_URL} /content/icy_tower

%cd /content/icy_tower
print("Katalog:", os.getcwd())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Path(DRIVE_DIR, "models").mkdir(parents=True, exist_ok=True)
Path(DRIVE_DIR, "logs").mkdir(parents=True, exist_ok=True)
print(f"Zapis: {DRIVE_DIR}")

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Trening od zera

Pomiń tę komórkę, jeśli kontynuujesz z `--load-model` poniżej.

In [ ]:
!python train.py \
  --n-envs {N_ENVS} \
  --timesteps {TIMESTEPS} \
  --n-eval-episodes 30 \
  --save-dir {DRIVE_DIR}/models \
  --tb-log {DRIVE_DIR}/logs \
  --run-name {RUN_NAME}

## Kontynuacja (po zerwaniu sesji)

Ustaw `ADDITIONAL_STEPS` — ile kroków **dokładamy** do wczytanego modelu.

In [ ]:
LOAD_MODEL = f"{DRIVE_DIR}/models/icy_ppo_final"  # lub .../best/best_model
ADDITIONAL_STEPS = 1_000_000

!python train.py \
  --load-model {LOAD_MODEL} \
  --additional-timesteps {ADDITIONAL_STEPS} \
  --n-envs {N_ENVS} \
  --n-eval-episodes 30 \
  --save-dir {DRIVE_DIR}/models \
  --tb-log {DRIVE_DIR}/logs \
  --run-name {RUN_NAME}_resume

## TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {DRIVE_DIR}/logs

## Pobierz model na PC (opcjonalnie)

Potem lokalnie: `python test.py --model ścieżka/do/icy_ppo_final`

In [ ]:
from google.colab import files
files.download(f"{DRIVE_DIR}/models/icy_ppo_final.zip")